# CPU：全量 EEG / MEG 缓存预热
选择纯 CPU 运行时，依次执行。源码只从 GitHub 获取，不嵌入源码包。
原始数据保留在 Drive；本流程调用同一套 `Experiment.data.build()`，不创建神经网络或 GPU Trainer。
首次全量预处理可能很久；重跑会复用已完成的 timeline 和信号缓存。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
REPO_URL = "https://github.com/wdongdong0708-jpg/brain2qwertyv1Test.git"
CODE = Path("/content/code")
WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
if (CODE / ".git").exists():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(CODE)], check=True)
os.chdir(CODE)
for folder in ("data", "cache_persisted", "archives", "runs"):
    (WORKSPACE / folder).mkdir(parents=True, exist_ok=True)
raw = WORKSPACE / "data/SpanishBCBL"
if not raw.is_dir():
    raise FileNotFoundError(f"Put the already downloaded full SpanishBCBL repository at {raw}")
for name in ("Pinet2024Eeg", "Pinet2024Meg"):
    alias = WORKSPACE / "data" / name
    if not alias.exists():
        try:
            alias.symlink_to(raw, target_is_directory=True)
        except OSError:
            print(f"Drive does not support this symlink; the reader will resolve {raw} directly")
os.environ["BRAIN2QWERTY_STUDIES"] = str(WORKSPACE / "data")
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [ ]:
%cd /content/code

# 1. 删掉刚才没装上 pip 的残缺环境，用 virtualenv 建一个自带最新 pip 的完整环境
!rm -rf /content/b2q-venv
!pip install -q virtualenv
!virtualenv --system-site-packages /content/b2q-venv

# 2. 重新执行 Codex 写的初始化脚本（此时已有 pip，并且保留 runtime-constraints 约束）
!/usr/bin/python3 -m scripts.colab_setup

PYTHON = "/content/b2q-venv/bin/python"
!{PYTHON} -c "import brain2qwerty_v1, studies; print('✅ /content/b2q-venv 环境安装大功告成！')"


In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
RAW_DIR = WORKSPACE / "data" / "SpanishBCBL"

# 1. 在 Colab 本地盘建一个专门放软链接的轻量目录（完美绕开 Google Drive 不支持软链接的限制）
LOCAL_STUDIES = Path("/content/studies_links")
LOCAL_STUDIES.mkdir(parents=True, exist_ok=True)

for study in ("Pinet2024Eeg", "Pinet2024Meg"):
    link_path = LOCAL_STUDIES / study
    if link_path.is_symlink() or link_path.exists():
        link_path.unlink()
    os.symlink(RAW_DIR, link_path)

print("✅ 本地软链接桥接成功：", list(LOCAL_STUDIES.iterdir()))

# 2. 缓存放本地高速盘，数据路径指向刚才建好的本地软链接桥
LOCAL_CACHE = Path("/content/local_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["BRAIN2QWERTY_STUDIES"] = str(LOCAL_STUDIES)
os.environ["BRAIN2QWERTY_CACHE"] = str(LOCAL_CACHE)
os.environ["BRAIN2QWERTY_RESULTS"] = str(WORKSPACE / "runs")

# 3. 正式启动全量 EEG 预热
CPU_THREADS = 2
print("⚙️ 开始预热全量 EEG 特征缓存...")
!{PYTHON} -u -m brain2qwerty_v1 cache --modality eeg --cpu-threads {CPU_THREADS}

可选：所有缓存完成后生成一个未压缩 tar。后续 GPU 实例顺序读取 tar 并解压到本地磁盘。打包时不要同时启动写入这些缓存的任务。


In [ ]:
from pathlib import Path

WORKSPACE = Path("/content/drive/MyDrive/Brain2Qwerty_Workspace")
ARCHIVE_DIR = WORKSPACE / "archives"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
PYTHON = "/content/b2q-venv/bin/python"

print("📦 正在将本地高速盘算好的 EEG 缓存打包存入 20TB 云盘...")
!{PYTHON} -u -m scripts.cache_archive pack \
    --cache-root /content/local_cache \
    --archive "{ARCHIVE_DIR}/features_eeg.tar"

!ls -lh "{ARCHIVE_DIR}/features_eeg.tar"
print("🎉 打包完成！可以随时开 GPU 实例秒级解压开训了！")
